# 02 – DPO: Kreativ-gültig bevorzugen
Vorher: Output von 01 als Input hinzufügen (**Add Input → Your Work → 01_sft**). Pfad unten anpassen.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"   # <- anpassen

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

SFT_ADAPTER = "/kaggle/input/01-sft/adapter_sft"   # <- an Input-Namen anpassen
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER, max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
# Adapter weiter trainierbar machen
for n, p in model.named_parameters():
    if "lora_" in n:
        p.requires_grad = True
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="repo/data/dpo.jsonl", split="train")
eos = tokenizer.eos_token
ds = ds.map(lambda r: {
    "prompt":   chat(tokenizer, [{"role":"user","content":r["prompt"]}], add_generation_prompt=True),
    "chosen":   r["chosen"] + eos,
    "rejected": r["rejected"] + eos})
print(len(ds))

In [ ]:
from trl import DPOTrainer, DPOConfig
D = CFG["dpo"]
trainer = DPOTrainer(
    model=model, ref_model=None, processing_class=tokenizer, train_dataset=ds,
    args=DPOConfig(
        beta=D["beta"], max_length=CFG["max_seq_length"], max_prompt_length=512,
        per_device_train_batch_size=D["batch_size"], gradient_accumulation_steps=D["grad_accum"],
        num_train_epochs=D["epochs"], learning_rate=D["lr"], warmup_ratio=0.1,
        optim="adamw_8bit", fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5, save_steps=200, save_total_limit=2,
        output_dir="checkpoints_dpo", report_to="none", seed=42))
trainer.train()
# Achte auf rewards/margins: sollte steigen; rewards/accuracies Richtung 0.8+

In [ ]:
model.save_pretrained("adapter_dpo"); tokenizer.save_pretrained("adapter_dpo")
print("Gespeichert in /kaggle/working/adapter_dpo → 'Save Version'")